# Train `multi_task_dit` (flow matching) on Colab

Cloud counterpart of the local run — same policy, a real GPU (bigger batch, faster). The pick-place dataset is pulled from the HF Hub (push it first from your Mac with `python -m pick_place.cloud.push_dataset --repo-id <you>/pick_place`).

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine for the DiT; A100 for smolvla/pi0).

**Edit one thing:** `REPO_ID` in the config cell.

> Note: locally this DiT plateaued fast (~300 demos is the data ceiling), so cloud's win here is *speed + headroom* (bigger batch, and the infra to later scale data / try smolvla / pi0), not magic quality. Still the right place to run the proper recipe.

In [ ]:
!nvidia-smi -L    # confirm a GPU is attached

In [ ]:
# deps: lerobot + training/DiT extras, and ffmpeg (torchcodec needs a system ffmpeg)
# Pin 0.6.0: newer PyPI lerobot dropped multi_task_dit (ships groot/xvla/sarm instead).
!apt-get -qq install -y ffmpeg
!pip -q install 'lerobot[dataset,training,multi_task_dit]==0.6.0'

In [ ]:
# log in so the private dataset can be pulled
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
REPO_ID = "YOUR_HF_USERNAME/pick_place"   #  <-- EDIT ME (the repo you pushed)

# Colab storage is ephemeral — mount Drive so checkpoints survive disconnects.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/pick_place/dit_flow'
else:
    OUTPUT_DIR = '/content/outputs/dit_flow'
print('checkpoints ->', OUTPUT_DIR)

In [ ]:
# batch_size is GPU-bound (CLIP x 3 cams x 2 obs-frames + AdamW states):
#   free T4 (~15 GB) -> 8   |   A100 (40/80 GB) -> 64+
# Optional to fit a bigger batch on T4: add --policy.use_amp=true (fp16, ~halves
#   activation memory) and/or --policy.n_obs_steps=1 (halves the CLIP passes).
# expandable_segments cuts fragmentation OOMs. Default LR (2e-5) is fine.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!lerobot-train \
  --dataset.repo_id={REPO_ID} \
  --policy.type=multi_task_dit --policy.objective=flow_matching --policy.device=cuda \
  --policy.push_to_hub=false \
  --batch_size=8 --steps=15000 --save_freq=3000 --num_workers=2 \
  --log_freq=100 --wandb.enable=false \
  --output_dir={OUTPUT_DIR}

## Get the trained policy back

- If you mounted Drive, checkpoints are already in `MyDrive/pick_place/dit_flow/checkpoints/` — download the latest `pretrained_model/` folder.
- Or push it to the Hub and pull on your Mac:
  ```python
  from lerobot.policies.multi_task_dit.modeling_multi_task_dit import MultiTaskDiTPolicy
  p = MultiTaskDiTPolicy.from_pretrained(f'{OUTPUT_DIR}/checkpoints/last/pretrained_model')
  p.push_to_hub('YOUR_HF_USERNAME/pick_place-dit')
  ```
- Then run inference **locally** (the MuJoCo sim/viz lives on your Mac):
  ```bash
  .venv/bin/mjpython -m pick_place.run_infer --viz live \
    --checkpoint <downloaded>/pretrained_model
  ```